[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C58_HardCase_LongTail_Course/05_close_loop/05_closed_loop_validation.ipynb)

# 05 · 闭环验证：证明数据真的有用（切片评测 / 显著性 / 回归门禁 / 配比 / **边际收益曲线** / 周期时间）

目标：把「加了 1200 张隧道口施工牌，模型变好了吗」这个问题，变成一套**可以自动跑、
会输出 PASS/FAIL、并且能告诉你还该不该继续标**的流程。

路线：合成评测集与「配比 → 分场景 AP」模拟器 → **切片评测**（平均值怎么骗人）→
配对检验 vs 非配对 + bootstrap → **多重比较**（20 个切片必然出假阳性）→
**回归门禁** → 数据配比扫描找 p\* → **边际收益曲线拟合与停标决策** → 闭环周期时间 →
✏️ 练习（功效分析 / BH 校正 / 停标规则 / 周期时间分解）→ 📖 答案 → 🧪 工程胶囊。

本 notebook 你会亲手实现：
- 一个「训练配比 → 各切片 AP」的模拟器（含**重复采样的信息上限**）
- 逐图配对检验（正态近似）与 bootstrap 置信区间，量化**配对能省多少样本量**
- 多重比较的家族错误率仿真：20 个切片、无真实差异时，**64% 的实验会「发现」假回归**
- 带容差分层 + Bonferroni 校正的**回归门禁**，输出 PASS / WARN / FAIL 与原因
- 新旧数据配比 p 的约束优化，找出可行域与 p\*
- **边际收益曲线** `AP(n) = A∞ − (A∞ − A₀)e^{−n/τ}` 的拟合、A∞ 的置信区间、
  「再标 B 张能涨多少 / 每个 AP 点多少钱 / 目标可不可达」
- 闭环流水线的周期时间仿真：p50/p90、瓶颈定位、**一次通过率的乘性杠杆**

> 心智模型：**整体 mAP 是加权平均，而加权平均就是用来掩盖长尾的。
> 闭环验证要做的，是把这个平均值拆回它掩盖掉的那些东西。**

## 1 · 合成评测集与「配比 → 分场景 AP」模拟器

模拟器的规则（**简化但方向正确**，先说清楚再用）：

1. 每个切片的 AP 由它在训练中的**有效样本量** `e` 决定：`AP = A∞ − (A∞ − A₀)·exp(−e/τ)`；
2. 训练的总曝光预算 `T` 固定（算力有限），各切片按采样配比瓜分；
3. **重复采样有上限**：把同 2100 张图重复 20 遍不会产生 20 倍的信息，
   所以 `e = min(唯一样本数 × REPEAT_CAP, 该切片的曝光量)`。第 3 条会带来一个很有意思的结论。

In [ ]:
import numpy as np, math, collections, json
rng = np.random.default_rng(2026)

# 切片定义：n_eval 评测图数 / n_tr 当前训练样本数 / 饱和曲线三参数 / 门禁容差 / 是否安全关键
SLICES = {
    'day_clear_urban': dict(n_eval=520, n_tr=42000, a_inf=0.90, tau=5000, a0=0.20, tol=0.005, crit=False),
    'night_urban':     dict(n_eval=260, n_tr=16000, a_inf=0.84, tau=2600, a0=0.15, tol=0.005, crit=False),
    'rain_highway':    dict(n_eval=150, n_tr=9000,  a_inf=0.83, tau=1800, a0=0.15, tol=0.005, crit=False),
    'dusk_ramp':       dict(n_eval=110, n_tr=7000,  a_inf=0.81, tau=1600, a0=0.15, tol=0.005, crit=False),
    'fog_tunnel':      dict(n_eval=95,  n_tr=900,   a_inf=0.78, tau=2500, a0=0.30, tol=0.005, crit=True),
    'snow_any':        dict(n_eval=60,  n_tr=3000,  a_inf=0.76, tau=900,  a0=0.15, tol=0.015, crit=False),
}
TARGET = 'fog_tunnel'          # ← 本轮要修的失效场景：雾天隧道口的施工牌（模块 04 挖来的）
NEW_LABELS = 1200              # 新标注的样本数
REPEAT_CAP = 4                 # 同一张图重复超过 4 遍，基本不再带来新信息
T_TOTAL = sum(v['n_tr'] for v in SLICES.values())

def ap_curve(n, a_inf, tau, a0):
    return a_inf - (a_inf - a0) * np.exp(-np.asarray(n, float) / tau)

def effective_n(p):
    '''p = 目标切片在训练采样中的配比；其余切片按原比例瓜分 (1-p)。'''
    base_share = {s: v['n_tr'] / T_TOTAL for s, v in SLICES.items()}
    others = 1 - base_share[TARGET]
    out = {}
    for s, v in SLICES.items():
        if s == TARGET:
            out[s] = min((v['n_tr'] + NEW_LABELS) * REPEAT_CAP, p * T_TOTAL)   # ← 信息上限
        else:
            out[s] = base_share[s] * (1 - p) / others * T_TOTAL
    return out

def model_ap(p=None):
    '''p=None 表示 baseline（没加新数据）。'''
    if p is None:
        return {s: float(ap_curve(v['n_tr'], v['a_inf'], v['tau'], v['a0'])) for s, v in SLICES.items()}
    e = effective_n(p)
    return {s: float(ap_curve(e[s], v['a_inf'], v['tau'], v['a0'])) for s, v in SLICES.items()}

ap_base = model_ap(None)
print(f'总曝光预算 T = {T_TOTAL}，目标切片 {TARGET} 当前只有 {SLICES[TARGET]["n_tr"]} 张训练样本\n')
print(f'{"切片":<18s} {"评测图":>7s} {"训练样本":>9s} {"baseline AP":>12s}')
for s, v in SLICES.items():
    star = '  ← 目标切片（惨不忍睹）' if s == TARGET else ''
    print(f'{s:<18s} {v["n_eval"]:>7d} {v["n_tr"]:>9d} {ap_base[s]:>12.4f}{star}')
assert ap_base[TARGET] < 0.5 and ap_base['day_clear_urban'] > 0.88
print('\n✅ 典型的长尾格局：常见场景已经深度饱和，目标场景连 0.5 都不到。')

## 2 · 分场景切片评测：整体 mAP 是用来骗人的

先造两个候选模型：
- **候选 A**：把新数据配比拉到 `p = 0.30`（「稀有场景要加权」的朴素做法）；
- **候选 B**：`p = 0.08`（第 6 节会算出这个数字怎么来的）。

评测用**逐图分数**（可以理解为每张图的检测质量），baseline 与新模型在**同一批图**上评测，
共享「图像固有难度」——这一点在第 3 节做配对检验时至关重要。

In [ ]:
DIFF = {s: rng.normal(0, 0.10, v['n_eval']) for s, v in SLICES.items()}   # 图像固有难度（两模型共享）

def make_scores(apd, seed):
    r = np.random.default_rng(seed)
    return {s: np.clip(apd[s] + DIFF[s] + r.normal(0, 0.03, SLICES[s]['n_eval']), 0, 1) for s in SLICES}

S_base = make_scores(ap_base, 11)
S_A    = make_scores(model_ap(0.30), 12)      # 激进配比
S_B    = make_scores(model_ap(0.08), 13)      # 保守配比

def overall_map(S):
    return float(np.concatenate([S[s] for s in SLICES]).mean())   # 按图数加权 = 整体 mAP

print(f'整体 mAP:  baseline {overall_map(S_base):.4f}  ->  候选A {overall_map(S_A):.4f} '
      f'({100 * (overall_map(S_A) - overall_map(S_base)):+.2f} 点)')
print('「涨了 1.7 点，合入！」—— 现在把它拆开看：\n')
print(f'{"切片":<18s} {"n_eval":>7s} {"baseline":>9s} {"候选A":>9s} {"Δ(点)":>8s} {"权重":>7s}')
n_all = sum(v['n_eval'] for v in SLICES.values())
for s, v in SLICES.items():
    d = (S_A[s].mean() - S_base[s].mean()) * 100
    flag = '  ◄── **回归**' if d < -0.4 else ('  ◄── 目标切片' if s == TARGET else '')
    print(f'{s:<18s} {v["n_eval"]:>7d} {S_base[s].mean():>9.4f} {S_A[s].mean():>9.4f} '
          f'{d:>+8.2f} {v["n_eval"] / n_all:>7.1%}{flag}')

n_reg = sum(1 for s in SLICES if (S_A[s].mean() - S_base[s].mean()) * 100 < -0.4)
assert overall_map(S_A) > overall_map(S_base), '整体是涨的'
assert n_reg >= 3, '同时有多个切片在回归'
print(f'\n⚠️  整体 +{100 * (overall_map(S_A) - overall_map(S_base)):.2f} 点的背后，'
      f'是 **{n_reg} 个切片在回归**，其中一个是占 22% 权重的常见场景。')
print('    目标切片 +32 点乘以 8% 的权重 = +2.6 点的贡献，恰好把几个回归的坑填平了。')
print('✅ **整体 mAP 是按样本数加权的平均，而加权平均就是用来掩盖长尾的。**')
print('   报告模板必须把「预期收益切片」和「回归哨兵切片」物理分成两块，回归块放前面。')

## 3 · 配对检验：免费拿回一个数量级的统计功效

baseline 与新模型在**同一批图**上评测 → 误差高度相关。
对每张图算差值 `d_i = s_new,i − s_base,i` 再做统计，
「图像固有难度」这一大块方差被直接抵消掉。

In [ ]:
def norm_cdf(z):
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

def norm_ppf(q, lo=-12.0, hi=12.0):
    for _ in range(200):                       # 二分求逆（不用 scipy）
        m = (lo + hi) / 2
        if norm_cdf(m) < q: lo = m
        else:               hi = m
    return (lo + hi) / 2

def paired_test(d):
    '''配对检验：对逐图差值做单样本 z 检验（n 大时正态近似足够）。'''
    d = np.asarray(d, float); n = len(d)
    mean, sd = float(d.mean()), float(d.std(ddof=1))
    se = sd / math.sqrt(n) if sd > 0 else 1e-12
    z = mean / se
    return dict(mean=mean, sd=sd, se=se, z=z, p=2 * (1 - norm_cdf(abs(z))), n=n)

def unpaired_test(a, b):
    '''非配对：把两组当成独立样本 —— 图像固有难度被算进了噪声。'''
    a, b = np.asarray(a, float), np.asarray(b, float)
    se = math.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    z = (b.mean() - a.mean()) / se
    return dict(mean=float(b.mean() - a.mean()), se=se, z=z, p=2 * (1 - norm_cdf(abs(z))))

s = 'night_urban'
pt = paired_test(S_A[s] - S_base[s])
ut = unpaired_test(S_base[s], S_A[s])
print(f'切片 {s}（n={pt["n"]}），同一个 Δ = {pt["mean"] * 100:+.2f} 点：')
print(f'  配对检验     SE={pt["se"]:.5f}  z={pt["z"]:+.2f}  p={pt["p"]:.4f}')
print(f'  非配对检验   SE={ut["se"]:.5f}  z={ut["z"]:+.2f}  p={ut["p"]:.4f}')
print(f'  -> 配对把标准误缩小了 {ut["se"] / pt["se"]:.1f}×，等价于样本量放大 {(ut["se"] / pt["se"]) ** 2:.0f}×')
assert pt['se'] < ut['se'] / 2 and pt['p'] < ut['p']
print('\n⚠️  非配对检验说「p=0.46，不显著」—— 但那是**错的结论**：')
print('    它把「有些图天生就难」也算进了噪声。评测集是固定的，那部分方差本该被消掉。')
print('✅ 只要两个模型评的是同一批图，就**必须**用配对统计。这是零成本的功效提升。')

In [ ]:
def bootstrap_ci(d, B=4000, alpha=0.05, seed=0):
    '''对图像重采样，给出 Δ 的置信区间（不依赖正态假设）。'''
    d = np.asarray(d, float); r = np.random.default_rng(seed); n = len(d)
    means = np.array([d[r.integers(0, n, n)].mean() for _ in range(B)])
    return float(np.percentile(means, 100 * alpha / 2)), float(np.percentile(means, 100 * (1 - alpha / 2)))

print(f'{"切片":<18s} {"Δ(点)":>8s} {"95% CI(点)":>20s} {"p(配对)":>10s} {"结论"}')
for s in SLICES:
    d = S_A[s] - S_base[s]
    lo, hi = bootstrap_ci(d, seed=hash(s) % 9999)
    t = paired_test(d)
    concl = '显著变化' if hi < 0 or lo > 0 else '落在噪声带内'
    print(f'{s:<18s} {t["mean"] * 100:>+8.2f} [{lo * 100:>+7.2f}, {hi * 100:>+7.2f}] '
          f'{t["p"]:>10.2e} {concl}')

d_day = S_A['day_clear_urban'] - S_base['day_clear_urban']
lo, hi = bootstrap_ci(d_day, seed=1)
assert lo < 0 < hi, 'day_clear_urban 的变化应落在噪声带内（区间跨 0）'
print('\n✅ 置信区间比「p 值 + 一句显著/不显著」信息量大得多：')
print('   它同时告诉你**方向、幅度、以及不确定性**。周报里只写 Δ 不写 CI 是不合格的。')
print('⚠️  这里的 CI 只覆盖了**评测集抽样噪声**。训练种子的噪声（检测任务典型 ±0.2~0.5 点）')
print('    必须靠真的重训 3~5 次才能量化 —— 这是行业里最普遍的方法论债务。')

## 4 · 多重比较：切片一多，假阳性是必然的

假设新模型**完全没有变化**。20 个切片、α=0.05 时，
至少出现一个「显著」的概率是 `1 − 0.95²⁰ ≈ 64%`。
也就是说：只要切片够多，每次实验都能「发现」一个回归——**而它是假的**。

In [ ]:
def fwer_sim(m=20, n=150, trials=300, alpha=0.05, seed=5):
    '''仿真：m 个切片、真实差异全为 0，看「至少一个假阳性」的概率。'''
    r = np.random.default_rng(seed)
    raw = bonf = 0
    for _ in range(trials):
        ps = np.array([paired_test(r.normal(0, 0.042, n))['p'] for _ in range(m)])
        raw  += int((ps < alpha).any())
        bonf += int((ps < alpha / m).any())
    return raw / trials, bonf / trials

print(f'{"切片数 m":>8s} {"理论 FWER":>11s} {"仿真 不校正":>13s} {"仿真 Bonferroni":>16s}')
for m in [1, 5, 20]:
    f_raw, f_bonf = fwer_sim(m=m, trials=250, seed=5 + m)
    print(f'{m:>8d} {1 - 0.95 ** m:>11.1%} {f_raw:>13.1%} {f_bonf:>16.1%}')

f_raw20, f_bonf20 = fwer_sim(m=20, trials=300, seed=25)
assert f_raw20 > 0.4, '不校正时假阳性率应该很高'
assert f_bonf20 < 0.15, 'Bonferroni 应把家族错误率压回 α 附近'
print('\n⚠️  **「我们切了 20 个维度，发现夜间场景显著回归了」—— 这句话在统计上什么也没说。**')
print('✅ 两条对策，用途不同：')
print('   · Bonferroni（阈值除以 m）控制 FWER —— **回归门禁用它**，宁可漏报也别乱报警。')
print('   · Benjamini-Hochberg 控制 FDR —— 探索性分析用它（练习 2 会实现）。')
print('   · 最有效的其实是**预注册**：跑实验前先声明主指标与哨兵切片，事后翻表才需要校正。')

## 5 · 回归门禁：把「不掉点」从自觉变成强制

判定逻辑（每个切片独立跑）：

| 条件 | 判定 |
|---|---|
| `Δ ≥ −tol` | PASS（在容差内） |
| `Δ < −tol` 且 `p > α/m` | WARN（掉了但不显著） |
| `Δ < −tol` 且 `p ≤ α/m` | **FAIL** |

容差按切片分层：安全关键切片 `tol = 0`，常见切片 `0.005`，小样本切片 `0.015`。

In [ ]:
def regression_gate(S_base, S_new, alpha=0.05, verbose=True):
    m = len(SLICES)
    rows = []
    for s, v in SLICES.items():
        t = paired_test(S_new[s] - S_base[s])
        tol = 0.0 if v['crit'] else v['tol']
        if   t['mean'] >= -tol:        verdict = 'PASS'
        elif t['p'] > alpha / m:       verdict = 'WARN'
        else:                          verdict = 'FAIL'
        rows.append(dict(slice=s, delta=t['mean'], p=t['p'], tol=tol,
                         n=t['n'], verdict=verdict))
    overall = ('FAIL' if any(r['verdict'] == 'FAIL' for r in rows) else
               'WARN' if any(r['verdict'] == 'WARN' for r in rows) else 'PASS')
    if verbose:
        print(f'{"切片":<18s} {"Δ(点)":>8s} {"容差(点)":>9s} {"p":>10s} {"判定":>6s}')
        for r in sorted(rows, key=lambda r: r['delta']):
            print(f'{r["slice"]:<18s} {r["delta"] * 100:>+8.2f} {r["tol"] * 100:>9.1f} '
                  f'{r["p"]:>10.2e} {r["verdict"]:>6s}')
        print(f'  -> 总判定 **{overall}**   (Bonferroni α\' = {alpha / m:.4f})')
    return rows, overall

print('== 候选 A（p = 0.30，激进配比）==')
rows_A, ver_A = regression_gate(S_base, S_A)
assert ver_A == 'FAIL'
n_fail = sum(1 for r in rows_A if r['verdict'] == 'FAIL')
n_warn = sum(1 for r in rows_A if r['verdict'] == 'WARN')
print(f'\n❌ 整体 mAP 涨 1.8 点的候选 A 被门禁拦下：{n_fail} 个切片 FAIL、{n_warn} 个 WARN。')
print('⚠️  门禁最大的敌人不是统计学，是**人**：一个天天误报的门禁，三周内就会长出')
print('    `--skip-gate` 参数。所以第一设计目标是「误报率低到 FAIL 出现时所有人都信」，')
print('    这比「不漏报」更重要 —— 宁可只守住真正要命的几个切片。')
print('⚠️  门禁必须跑在**不参与本轮挖掘**的固定基准集上，否则就是自己给自己出题。')

## 6 · 数据配比：最优点是内点，而且有一个「天花板」

扫描配比 `p`，看两件事：目标切片能涨到多少、旧切片最多掉多少。
约束优化：**在所有旧切片都不超过各自容差的前提下，最大化目标切片 AP**。

In [ ]:
def scan_ratio(ps):
    out = []
    for p in ps:
        a = model_ap(p)
        viol = max((ap_base[s] - a[s]) - v['tol'] for s, v in SLICES.items() if s != TARGET)
        worst = max(((ap_base[s] - a[s]) / max(v['tol'], 1e-9), s)
                    for s, v in SLICES.items() if s != TARGET)[1]
        out.append(dict(p=p, ap_target=a[TARGET], viol=viol, worst=worst, feasible=viol <= 0))
    return out

grid = scan_ratio(np.linspace(0.005, 0.40, 160))
feas = [r for r in grid if r['feasible']]
best_ap = max(r['ap_target'] for r in feas)
p_star = min(r['p'] for r in feas if r['ap_target'] >= best_ap - 1e-9)

print(f'{"p":>8s} {"目标切片 AP":>12s} {"最差旧切片超额":>15s} {"绑定的切片":>14s} {"可行":>6s}')
for r in scan_ratio([0.02, 0.05, 0.08, 0.11, 0.13, 0.20, 0.30]):
    print(f'{r["p"]:>8.3f} {r["ap_target"]:>12.4f} {r["viol"] * 100:>+15.2f} '
          f'{r["worst"]:>14s} {"yes" if r["feasible"] else "**NO**":>6s}')
print(f'\n可行域内的最优配比 p* = {p_star:.3f}，目标切片 AP = {best_ap:.4f}')

cap = (SLICES[TARGET]['n_tr'] + NEW_LABELS) * REPEAT_CAP
print(f'\n💡 注意 p >= {cap / T_TOTAL:.3f} 之后目标切片 AP **完全不再上升** ——')
print(f'   因为有效样本量被 唯一样本数×REPEAT_CAP = {cap} 卡死了。')
print('   **重复采样不创造信息。** 再往上调配比，只剩下对旧场景的伤害。')
assert 0.02 < p_star < 0.30, '最优配比是内点，不是端点'
assert abs(model_ap(0.20)[TARGET] - model_ap(0.30)[TARGET]) < 1e-9, '超过上限后目标 AP 不再变化'

print('\n== 候选 B（p = 0.08，取可行域内侧留安全余量）==')
rows_B, ver_B = regression_gate(S_base, S_B)
assert ver_B == 'PASS'
print(f'\n✅ 候选 B 拿到了 {(S_B[TARGET].mean() - S_base[TARGET].mean()) * 100:+.1f} 点的目标切片提升，')
print('   而所有旧切片都在容差内 —— 同样一批数据，只是配比不同。')
print('⚠️  为什么取 0.08 而不是算出来的 p*？因为 p* 是在**无噪声的模型上**算的，')
print('    而门禁用的是**带噪声的观测值**。卡在可行域边界上，会随机地 WARN。')
print('    工程实践：**约束优化给上界，实际取内侧留 20~30% 余量。**')

## 7 · 边际收益曲线：还值不值得继续标

$$\mathrm{AP}(n) = A_\infty - (A_\infty - A_0)e^{-n/\tau}$$

拟合方法：对 τ 做一维网格搜索，固定 τ 后模型对 $(A_\infty, A_0)$ 是**线性**的，
用最小二乘闭式求解。这样只用 numpy 就能拟合三参数曲线。

In [ ]:
TRUE_CURVE = dict(a_inf=0.78, tau=2500, a0=0.30)     # 真实曲线（现实中当然不知道）
EVAL_NOISE = 0.005                                   # 每个点的评测+种子噪声

# **分批标注**：这就是为什么必须小批多轮 —— 一次标 5000 张，你永远拿不到这条曲线
ns_obs = np.array([0, 300, 700, 1400, 2600, 4500])
ap_obs = ap_curve(ns_obs, **TRUE_CURVE) + rng.normal(0, EVAL_NOISE, len(ns_obs))

def fit_saturating(ns, aps, n_tau=600):
    ns, aps = np.asarray(ns, float), np.asarray(aps, float)
    best = None
    for tau in np.exp(np.linspace(math.log(100), math.log(200000), n_tau)):
        e = np.exp(-ns / tau)
        A = np.stack([1 - e, e], axis=1)              # 固定 tau 后对 (a_inf, a0) 线性
        coef, *_ = np.linalg.lstsq(A, aps, rcond=None)
        sse = float(((A @ coef - aps) ** 2).sum())
        if best is None or sse < best[0]:
            best = (sse, float(coef[0]), float(coef[1]), float(tau))
    return dict(sse=best[0], a_inf=best[1], a0=best[2], tau=best[3])

fit = fit_saturating(ns_obs, ap_obs)
print(f'{"n(该场景训练样本)":>18s} {"实测 AP":>9s} {"拟合值":>9s}')
for n_, a_ in zip(ns_obs, ap_obs):
    print(f'{n_:>18d} {a_:>9.4f} {float(ap_curve(n_, fit["a_inf"], fit["tau"], fit["a0"])):>9.4f}')
print(f'\n拟合: A∞={fit["a_inf"]:.4f}  τ={fit["tau"]:.0f}  A₀={fit["a0"]:.4f}')
print(f'真值: A∞={TRUE_CURVE["a_inf"]:.4f}  τ={TRUE_CURVE["tau"]:.0f}  A₀={TRUE_CURVE["a0"]:.4f}')
assert abs(fit['a_inf'] - TRUE_CURVE['a_inf']) < 0.06, 'A∞ 应被大致恢复'
assert 0.5 < fit['tau'] / TRUE_CURVE['tau'] < 2.0

# **A∞ 必须带置信区间**：用它做决策时要用区间下界，别用点估计
boot = np.array([fit_saturating(ns_obs, ap_obs + rng.normal(0, EVAL_NOISE, len(ns_obs)),
                                n_tau=200)['a_inf'] for _ in range(150)])
lo, hi = np.percentile(boot, [5, 95])
print(f'A∞ 的 90% 区间: [{lo:.4f}, {hi:.4f}]   -> 决策时用下界 {lo:.4f}')
assert lo < fit['a_inf'] < hi and hi - lo > 0.005, 'A∞ 的不确定性不该被忽略'
print(f'\n⚠️  注意：拟合的 A∞ 比真值高了 {(fit["a_inf"] - TRUE_CURVE["a_inf"]) * 100:+.1f} 点，'
      f'而 90% 区间甚至没盖住真值。')
print(f'    原因很具体：最大观测点 n={ns_obs[-1]} 只有 {ns_obs[-1] / TRUE_CURVE["tau"]:.1f}τ，'
      '**还没真正进入饱和区**，')
print('    此时外推天花板会**系统性偏乐观**。这正是本模块最后一节列的开放问题之一。')
print('    工程对策：① 决策用区间下界；② 天花板越关键，越要在大 n 处再补一个点。')
print('\n⚠️  拟合至少要 4~5 个点、且横跨一个数量级（300 / 700 / 1400 / 2600 / 4500），')
print('    只有两个点拟合不出饱和形状。**这意味着标注必须分批做、每批重训并评测。**')
print('⚠️  曲线只在「其他条件不变」时成立：中途换 backbone 或改增强，曲线要重拟。')

In [ ]:
def predict(f, n):
    return f['a_inf'] - (f['a_inf'] - f['a0']) * math.exp(-n / f['tau'])

def marginal(f, n, batch):
    '''再标 batch 张的预期 AP 提升（闭式）。'''
    return (f['a_inf'] - predict(f, n)) * (1 - math.exp(-batch / f['tau']))

def n_for_target(f, target):
    '''达到 target 需要多少样本；target >= A∞ 时返回 inf（**不可达**）。'''
    if target >= f['a_inf']:
        return float('inf')
    return -f['tau'] * math.log((f['a_inf'] - target) / (f['a_inf'] - f['a0']))

PRICE = 2.6            # 元 / 张（夜间+遮挡的标注单价）
n_now = int(ns_obs[-1])
print(f'当前该场景已有 {n_now} 张，AP = {predict(fit, n_now):.4f}\n')
print(f'{"再标":>8s} {"预期 ΔAP(点)":>13s} {"每千张(点)":>12s} {"花费(元)":>10s} {"每 AP 点(元)":>13s} {"建议":>10s}')
for B in [1000, 2000, 5000, 20000]:
    g = marginal(fit, n_now, B) * 100
    rec = '继续标' if g / B * 1000 >= 0.5 else '**停止**'
    print(f'{B:>8d} {g:>+13.2f} {g / B * 1000:>12.2f} {B * PRICE:>10.0f} '
          f'{B * PRICE / max(g, 1e-9):>13.0f} {rec:>10s}')

print(f'\n{"产品目标":>10s} {"需要样本数":>12s} {"结论"}')
for t in [0.70, 0.75, 0.78, 0.85]:
    n_ = n_for_target(fit, t)
    txt = f'{n_:,.0f} 张' if math.isfinite(n_) else '**不可达**'
    concl = '' if math.isfinite(n_) else '  ← 加再多数据也到不了，必须换方法'
    print(f'{t:>10.2f} {txt:>12s}{concl}')
assert not math.isfinite(n_for_target(fit, 0.85)), '目标高于 A∞ 时必须判定为不可达'
assert n_for_target(fit, 0.75) > n_for_target(fit, 0.70) > 0

print('\n✅ 这条曲线最大的作用：**把「还标不标」从立场之争变成算术题**。')
print(f'   A∞ = {fit["a_inf"]:.3f}，产品目标 0.85 -> 没什么可争的，继续标注不可能达标，')
print('   讨论应该立刻转向「换什么方法」：改架构 / 提分辨率 / 拆两级 / 换传感器。')
print('   （对 TSR 的雾天隧道口小目标而言，最可能有效的是提高输入分辨率与切片推理，见 C57。）')

## 8 · 闭环流水线：周期时间、瓶颈与「一次通过率」的乘性杠杆

闭环系统真正该被优化的指标，不是任何一个模型指标，
而是**从发现问题到修复上线的天数**——因为长尾问题是源源不断的。

In [ ]:
STAGES = [('触发', 0.5, 0.2), ('挖掘', 1.0, 0.4), ('精选', 0.5, 0.2), ('标注', 9.0, 3.5),
          ('训练', 2.5, 0.8), ('评测', 0.6, 0.2), ('门禁', 0.3, 0.1)]
RELEASE = ('发布', 4.0, 1.5)
MAX_ATTEMPTS = 3

def simulate_cycles(gate_pass_rate, trials=2000, seed=3):
    '''门禁 FAIL -> 退回「挖掘/精选/标注」重来一轮（返工）。'''
    r = np.random.default_rng(seed)
    totals, per_stage = [], collections.defaultdict(float)
    reworks = 0
    for _ in range(trials):
        t, attempts = 0.0, 0
        while True:
            attempts += 1
            for nm, mu, sd in STAGES:
                dt = max(0.05, r.normal(mu, sd)); t += dt; per_stage[nm] += dt
            if r.random() < gate_pass_rate or attempts >= MAX_ATTEMPTS:
                break
        reworks += (attempts > 1)
        dt = max(0.05, r.normal(*RELEASE[1:])); t += dt; per_stage[RELEASE[0]] += dt
        totals.append(t)
    totals = np.array(totals)
    tot_time = sum(per_stage.values())
    return dict(p50=float(np.percentile(totals, 50)), p90=float(np.percentile(totals, 90)),
                mean=float(totals.mean()), rework_rate=reworks / trials,
                share={k: v / tot_time for k, v in per_stage.items()})

r60 = simulate_cycles(0.60)
print(f'门禁一次通过率 60%:  p50 {r60["p50"]:.1f} 天   p90 {r60["p90"]:.1f} 天   '
      f'均值 {r60["mean"]:.1f} 天   返工率 {r60["rework_rate"]:.0%}')
print(f'\n{"阶段":<8s} {"占总耗时":>9s}')
for k, v in sorted(r60['share'].items(), key=lambda kv: -kv[1]):
    bar = '█' * int(v * 60)
    print(f'{k:<8s} {v:>9.1%}  {bar}')
bottleneck = max(r60['share'], key=r60['share'].get)
assert bottleneck == '标注', '标注通常是闭环的瓶颈'
assert r60['p90'] > r60['p50'] * 1.3, 'p90 才是真正的痛点'

print(f'\n{"一次通过率":>10s} {"均值周期":>10s} {"p90":>8s} {"返工率":>8s}')
for pr in [0.45, 0.60, 0.85]:
    r_ = simulate_cycles(pr, seed=3)
    print(f'{pr:>10.0%} {r_["mean"]:>10.1f} {r_["p90"]:>8.1f} {r_["rework_rate"]:>8.0%}')
r85 = simulate_cycles(0.85, seed=3)
assert r85['mean'] < r60['mean'], '提高一次通过率能显著缩短周期'
print(f'\n✅ 一次通过率 60% -> 85%，均值周期从 {r60["mean"]:.1f} 天降到 {r85["mean"]:.1f} 天。')
print('   **返工的影响是乘性的**：期望轮数 = 1/通过率。提高通过率的杠杆，')
print('   比把标注速度提升 30% 还大，而且便宜得多。')
print('   具体手段：训练前先跑配比预演、标注前小批试标验规范、门禁前先跑轻量自检。')
print('⚠️  另一个反直觉的结论：**缩短周期常常要靠「减少每轮的雄心」**。')
print('   一轮同时修 5 个失效模式，任何一个出问题整轮都卡住，且归因困难。')

## ✏️ 练习 1：功效分析 —— 「要检出 +0.3 点，需要多少样本」

实现 `n_needed(delta, sd, alpha=0.05, power=0.8)`：配对设计下，
要以 `power` 的概率检出大小为 `delta` 的真实差异，需要多少个配对样本。

$$n = \left\lceil \frac{(z_{1-\alpha/2} + z_{\text{power}})^2\, s_d^2}{\delta^2} \right\rceil$$

其中 `sd` 是**逐图差值 d 的标准差**（不是分数本身的标准差）。用上面写好的 `norm_ppf`。

In [ ]:
def n_needed(delta, sd, alpha=0.05, power=0.8):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
SD_D = float((S_A['night_urban'] - S_base['night_urban']).std(ddof=1))
n1 = n_needed(0.003, 0.042)
n2 = n_needed(0.006, 0.042)
n3 = n_needed(0.003, 0.042, power=0.9)
assert 1490 <= n1 <= 1590, n1
assert abs(n2 - n1 / 4) <= 3, (n1, n2)            # delta 翻倍 -> 样本量降到 1/4
assert n3 > n1 and isinstance(n1, int)
assert n_needed(0.003, 0.021) < n1                # 噪声减半 -> 样本量降到 1/4
print(f'逐图差值的实测标准差 sd_d = {SD_D:.4f}\n')
print(f'{"要检出的 Δ(点)":>14s} {"power=0.8":>11s} {"power=0.9":>11s}')
for dpt in [0.1, 0.3, 0.5, 1.0, 2.0]:
    print(f'{dpt:>14.1f} {n_needed(dpt / 100, SD_D):>11d} {n_needed(dpt / 100, SD_D, power=0.9):>11d}')
print('\n✅ 练习 1 通过：**「+0.3 点算不算提升」的答案，取决于你的评测集有多大。**')
print('   把这张表贴在评测集设计文档里，比事后争论有用得多。')

## ✏️ 练习 2：Benjamini–Hochberg FDR 校正

实现 `benjamini_hochberg(pvals, q=0.05)`：返回与 `pvals` 等长的**布尔数组**，
表示每个假设是否被拒绝（判为显著）。

算法：把 p 值升序排列得到 `p₍₁₎ ≤ … ≤ p₍ₘ₎`，
找出**最大**的 `k` 使得 `p₍ₖ₎ ≤ k·q/m`，然后拒绝排名前 `k` 的全部假设
（注意：是「前 k 个全拒」，不是「逐个比较」——这是最容易写错的地方）。
若不存在这样的 `k`，一个都不拒。

In [ ]:
def benjamini_hochberg(pvals, q=0.05):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ps = np.array([0.001, 0.008, 0.039, 0.041, 0.042, 0.060, 0.500])
rej = benjamini_hochberg(ps, 0.05)
assert list(rej) == [True, True, False, False, False, False, False], list(rej)
assert benjamini_hochberg(ps, 0.05).sum() > (ps < 0.05 / len(ps)).sum(), 'BH 应比 Bonferroni 宽松'
assert benjamini_hochberg(np.ones(10), 0.05).sum() == 0
assert benjamini_hochberg(np.full(10, 0.001), 0.05).all()
assert benjamini_hochberg(np.array([0.02, 0.03, 0.04]), 0.05).all(), '「前 k 个全拒」：逐个比会漏掉 0.02'

p_slices = np.array([r['p'] for r in rows_A])
names = [r['slice'] for r in rows_A]
bh = benjamini_hochberg(p_slices, 0.05)
bonf = p_slices < 0.05 / len(p_slices)
print(f'{"切片":<18s} {"p":>10s} {"Bonferroni":>11s} {"BH(q=.05)":>11s}')
for nm, p_, b1, b2 in zip(names, p_slices, bonf, bh):
    print(f'{nm:<18s} {p_:>10.2e} {str(bool(b1)):>11s} {str(bool(b2)):>11s}')
print('\n✅ 练习 2 通过：**门禁用 Bonferroni（怕误报），探索用 BH（怕漏掉线索）** ——')
print('   选哪个取决于你更怕哪种错，而不是哪个更「先进」。')

## ✏️ 练习 3：停标决策 —— 边际收益曲线的直接应用

实现 `label_more(fit, n_now, batch, price, min_gain_per_1k, target=None)`，返回一个 dict：

- `gain_points`：再标 `batch` 张的预期 AP 提升（**单位是「点」，即 ×100**）
- `gain_per_1k`：每千张的预期提升（点）
- `cost` / `cost_per_point`：花费与每个 AP 点的成本（元）
- `decision` ∈ `'continue'` / `'stop_diminishing'` / `'stop_unreachable'`

判定顺序（**先判天花板**）：① `target` 给定且 `target ≥ fit['a_inf']` → `stop_unreachable`；
② `gain_per_1k < min_gain_per_1k` → `stop_diminishing`；③ 否则 `continue`。

In [ ]:
def label_more(fit, n_now, batch, price, min_gain_per_1k, target=None):
    # TODO: 复用上面的 predict / marginal
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
FIT_DEMO = dict(a_inf=0.78, tau=2500, a0=0.30)
r1 = label_more(FIT_DEMO, 2000, 2000, 2.6, min_gain_per_1k=0.5)
assert abs(r1['gain_points'] - 11.88) < 0.15, r1['gain_points']
assert abs(r1['gain_per_1k'] - 5.94) < 0.10 and r1['decision'] == 'continue'
assert abs(r1['cost'] - 5200) < 1e-6 and abs(r1['cost_per_point'] - 437.7) < 6
r2 = label_more(FIT_DEMO, 12000, 2000, 2.6, min_gain_per_1k=0.5)
assert r2['decision'] == 'stop_diminishing', r2
r3 = label_more(FIT_DEMO, 2000, 2000, 2.6, min_gain_per_1k=0.5, target=0.80)
assert r3['decision'] == 'stop_unreachable', '天花板要先判：目标高于 A∞ 时，涨得再快也没用'
print(f'{"已有样本":>9s} {"再标":>7s} {"ΔAP(点)":>9s} {"每千张":>8s} {"每 AP 点(元)":>13s} {"决策":>18s}')
for n_ in [500, 2000, 5000, 12000, 30000]:
    r = label_more(FIT_DEMO, n_, 2000, 2.6, 0.5)
    print(f'{n_:>9d} {2000:>7d} {r["gain_points"]:>9.2f} {r["gain_per_1k"]:>8.2f} '
          f'{r["cost_per_point"]:>13.0f} {r["decision"]:>18s}')
print('\n✅ 练习 3 通过：**「先判天花板，再判边际」这个顺序是关键** ——')
print('   目标不可达时，边际收益再高也应该停下来换方法，而不是接着标。')

## ✏️ 练习 4：周期时间分解与瓶颈定位

实现 `cycle_time_breakdown(events)`：`events` 是 `(ticket, stage, start_day, end_day)` 的列表。
返回 dict，包含：

- `cycle_time`：`{ticket: 该 ticket 的最大 end − 最小 start}`
- `p50` / `p90`：所有 ticket 周期时间的分位数（用 `np.percentile`）
- `stage_share`：`{stage: 该阶段总耗时 / 全部阶段总耗时}`
- `bottleneck`：占比最大的阶段名
- `rework_rate`：**有任一阶段出现超过一次**的 ticket 占比

In [ ]:
EVENTS = [
    ('T1', '触发', 0, 1), ('T1', '挖掘', 1, 2), ('T1', '标注', 2, 12),
    ('T1', '训练', 12, 15), ('T1', '门禁', 15, 15.5), ('T1', '发布', 15.5, 20),
    ('T2', '触发', 0, 1), ('T2', '挖掘', 1, 3), ('T2', '标注', 3, 17),
    ('T2', '训练', 17, 20), ('T2', '门禁', 20, 20.5),
    ('T2', '标注', 20.5, 28), ('T2', '训练', 28, 31),        # ← 门禁 FAIL 后的返工
    ('T2', '门禁', 31, 31.5), ('T2', '发布', 31.5, 36),
    ('T3', '触发', 0, 0.5), ('T3', '挖掘', 0.5, 1.5), ('T3', '标注', 1.5, 9),
    ('T3', '训练', 9, 11), ('T3', '门禁', 11, 11.5), ('T3', '发布', 11.5, 14),
    ('T4', '触发', 0, 1), ('T4', '挖掘', 1, 2), ('T4', '标注', 2, 20),
    ('T4', '训练', 20, 24), ('T4', '门禁', 24, 24.5), ('T4', '发布', 24.5, 30),
]

def cycle_time_breakdown(events):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
br = cycle_time_breakdown(EVENTS)
assert br['cycle_time'] == {'T1': 20.0, 'T2': 36.0, 'T3': 14.0, 'T4': 30.0}, br['cycle_time']
assert abs(br['p50'] - 25.0) < 1e-6 and abs(br['p90'] - 34.2) < 1e-6, (br['p50'], br['p90'])
assert br['bottleneck'] == '标注'
assert abs(br['stage_share']['标注'] - 0.57) < 1e-6, br['stage_share']['标注']
assert abs(br['rework_rate'] - 0.25) < 1e-9
assert abs(sum(br['stage_share'].values()) - 1.0) < 1e-9
print(f'周期时间: p50 {br["p50"]:.1f} 天   p90 {br["p90"]:.1f} 天   返工率 {br["rework_rate"]:.0%}\n')
print(f'{"阶段":<8s} {"占比":>8s}')
for k, v in sorted(br['stage_share'].items(), key=lambda kv: -kv[1]):
    print(f'{k:<8s} {v:>8.1%}  {"█" * int(v * 50)}')
print(f'\n瓶颈: **{br["bottleneck"]}**')
print('✅ 练习 4 通过：**优化非瓶颈阶段的收益接近零** —— 训练加速 2× 只省 0.6 天，')
print('   而把标注周期缩短 30% 能省 5 天以上。先测量，再优化。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def n_needed(delta, sd, alpha=0.05, power=0.8):
    z_a = norm_ppf(1 - alpha / 2)
    z_b = norm_ppf(power)
    return int(math.ceil((z_a + z_b) ** 2 * sd ** 2 / delta ** 2))

In [ ]:
# 练习 2 参考答案
def benjamini_hochberg(pvals, q=0.05):
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(p)                       # 升序
    thresh = (np.arange(1, m + 1) * q) / m
    passing = np.where(p[order] <= thresh)[0]
    rej = np.zeros(m, dtype=bool)
    if len(passing):
        k = passing.max() + 1                   # **最大的 k**，然后前 k 个全拒
        rej[order[:k]] = True
    return rej

In [ ]:
# 练习 3 参考答案
def label_more(fit, n_now, batch, price, min_gain_per_1k, target=None):
    gain_points = marginal(fit, n_now, batch) * 100
    gain_per_1k = gain_points / batch * 1000
    cost = batch * price
    out = dict(gain_points=gain_points, gain_per_1k=gain_per_1k, cost=cost,
               cost_per_point=cost / max(gain_points, 1e-12))
    if target is not None and target >= fit['a_inf']:
        out['decision'] = 'stop_unreachable'    # ← 先判天花板
    elif gain_per_1k < min_gain_per_1k:
        out['decision'] = 'stop_diminishing'
    else:
        out['decision'] = 'continue'
    return out

In [ ]:
# 练习 4 参考答案
def cycle_time_breakdown(events):
    by_ticket = collections.defaultdict(list)
    for t, st, s0, s1 in events:
        by_ticket[t].append((st, float(s0), float(s1)))
    cycle = {t: max(e[2] for e in v) - min(e[1] for e in v) for t, v in by_ticket.items()}
    stage = collections.defaultdict(float)
    for t, st, s0, s1 in events:
        stage[st] += float(s1) - float(s0)
    total = sum(stage.values())
    rework = sum(1 for v in by_ticket.values()
                 if max(collections.Counter(e[0] for e in v).values()) > 1)
    ct = np.array(list(cycle.values()), dtype=float)
    return dict(cycle_time=cycle,
                p50=float(np.percentile(ct, 50)), p90=float(np.percentile(ct, 90)),
                stage_share={k: v / total for k, v in stage.items()},
                bottleneck=max(stage, key=stage.get),
                rework_rate=rework / len(by_ticket))

---
## 🧪 真实工程胶囊：一份可直接照搬的闭环验证配置与报告模板

In [ ]:
RECIPE = r'''
# ===================== closed_loop.yaml =====================
experiment_id: exp_2026w32_tunnel_construction
ticket:        TSR-4471                 # 与模块 04 的挖掘任务同源，全程可追溯

# ---- ① 预注册（跑实验**之前**填，之后不许改）----
preregistration:
  primary_metric:   fog_tunnel.AP       # 主指标：只此一个，不需要多重比较校正
  expected_gain:    ">= +8.0 点"        # 预期效应量。没达到 = 假设被证伪，也是有效结论
  regression_sentinels:                 # 哨兵切片：只有这几个进 Bonferroni 家族
    - day_clear_urban
    - night_urban
    - rain_highway
  exploratory: "其余切片仅作探索，用 BH(q=0.05)，结论需再验证"

# ---- ② 数据与配比 ----
data:
  base_dataset:   v1.7.2
  new_batches:    [b_tunnel_2026w31]    # 1204 帧
  mix_ratio_p:    0.08                  # **约束优化给上界 0.11，实际取内侧留余量**
  leakage_check:  {frame_id: strict, phash_thresh: 6, split_key: clip_id}
                                        # 与评测集交集非空 -> **直接终止实验**，不是警告
  seeds: [0, 1, 2]                      # 关键发布 3 种子；日常迭代 1 种子 + 更大容差

# ---- ③ 评测：两套集合，用途不能混 ----
eval_sets:
  benchmark:   tsr_bench_v6      # **固定基准集**：按部署真实分布随机采样，冻结
                                 #   -> 回归门禁的唯一依据
  failure_modes: tsr_fm_v12      # 失效模式集：随挖掘扩充 -> 只看目标场景修好没有
  slice_labels_version: 2026.03  # **切片标签冻结** —— 尺子不能跟着被测物一起变

# ---- ④ 回归门禁 ----
gate:
  test: paired_bootstrap         # 同一批图 -> 必须配对
  correction: bonferroni         # 门禁怕误报
  alpha: 0.05
  tolerance:
    safety_critical: 0.000       # stop / yield / 施工 / 限速下调 -> **零容忍**
    common:          0.005
    small_sample:    0.015       # n < 100 的切片，噪声本来就大
  global_gates:
    - fp_per_km_increase <= 5%   # 工程指标，比 precision 更贴近路测体验
    - p99_latency_increase <= 0% # 模型换了可能变慢
    - flicker_rate_increase <= 0%# 时序稳定性：AP 涨但框闪烁 = 体验变差
  on_warn:  "允许发布，但强制建单跟踪；连续两轮 WARN 升级为 FAIL"
  on_fail:  "退回调配比或补数据；**不允许 --skip-gate**"

# ---- ⑤ 边际收益曲线（每个重点场景维护一条）----
marginal_curve:
  model: "AP(n) = A_inf - (A_inf - A0) * exp(-n / tau)"
  points: [[0, 0.301], [300, 0.352], [700, 0.410], [1400, 0.494],
           [2600, 0.593], [4500, 0.679]]
  fitted:  {A_inf: 0.785, tau: 2543, A0: 0.291, A_inf_ci90: [0.762, 0.809]}
  stop_rule:
    min_gain_per_1k_labels: 0.5   # 点/千张，低于此值停止标注
    max_cost_per_ap_point:  800   # 元，超过此值停止标注
    unreachable_if: "target >= A_inf_ci90_low"   # **用区间下界判天花板**

# ---- ⑥ 发布与闭环回流 ----
rollout: [shadow_7d, internal_fleet_14d, limited_odd_30d, full]
shadow:
  log: disagreement_frames        # 新旧模型分歧帧 -> **直接回流成下一轮挖掘的种子**
  note: "影子模式没有真值：分歧率下降 != 模型变好"
cycle_time_slo:
  p50_days: 21
  p90_days: 35
  gate_first_pass_rate: ">= 0.80"  # 返工是乘性成本：期望轮数 = 1/通过率
'''
print(RECIPE)
for token in ['preregistration', 'mix_ratio_p', 'leakage_check', 'split_key',
              'safety_critical', 'fp_per_km_increase', 'A_inf_ci90',
              'min_gain_per_1k_labels', 'disagreement_frames', 'gate_first_pass_rate']:
    assert token in RECIPE, token
print('✅ 配方覆盖：预注册 / 配比 / 泄漏检查 / 双评测集 / 分层容差 / 全局工程门禁 / '
      '边际收益曲线与停标规则 / 分阶段放开 / 影子回流 / 周期时间 SLO')

### 小结

- **加数据之后必须回答三个问题**：目标场景涨了吗？其他场景掉了吗？涨的部分是数据的功劳还是噪声？
  三问分别对应切片评测、回归门禁、显著性检验。
- **整体 mAP 是按样本数加权的平均，而加权平均就是用来掩盖长尾的**。
  实测：整体 +1.7 点的候选，背后是 3 个切片在回归，其中一个占 22% 的评测权重。
- **两个模型评的是同一批图 -> 必须用配对统计**。实测配对把标准误缩小 3.4×，
  等价于样本量放大 11×；非配对检验会给出「不显著」这个错误结论。
- **切片一多，假阳性是必然的**：20 个切片、无真实差异时，64% 的实验会「发现」假回归。
  门禁用 Bonferroni（怕误报），探索用 BH（怕漏线索），最有效的是**预注册**。
- **回归门禁的第一设计目标是「误报率低到 FAIL 出现时所有人都信」**，这比不漏报更重要。
  容差按样本量与安全等级分层；安全关键切片零容忍；门禁必须跑在**不参与挖掘的固定基准集**上。
- **新旧数据配比的最优点是内点**（典型 10%–30%）。两个端点都错：按自然比例 = 「加了没效果」，
  只用新数据微调 = 灾难性遗忘。而且**重复采样不创造信息**——超过 `唯一样本数 × 重复上限`
  之后，再提高配比只剩下对旧场景的伤害。
- **边际收益曲线 `AP(n) = A∞ − (A∞ − A₀)e^{−n/τ}` 是数据决策的核心工具**：
  `A∞` 回答「天花板够不够得着目标」（**先判这个**），`ΔAP/千张 × 单价`回答「每个 AP 点多少钱」。
  拟合要 4–5 个横跨一个数量级的点 -> **标注必须小批多轮**，且 `A∞` 要带置信区间、用下界决策。
- **闭环该被优化的指标是周期时间，不是任何模型指标**。瓶颈通常在标注（占 50%+），
  而**返工的影响是乘性的**（期望轮数 = 1/通过率）：一次通过率 60%→85%，均值周期少 5 天以上。
- **离线全绿不等于能上车**：影子模式 → 内部车队 → 有限 ODD 灰度 → 全量，
  每阶段有独立退出准则；影子的分歧帧**直接回流成下一轮挖掘的种子**——这条回边才是「闭环」。

C58 到此结束。下一站：**C59 · VLA 与感知接口** —— 把 TSR 的输出送进下游决策。